# Asistente RAG sobre los reglamentos de pregrado de EAFIT

**SI3003 Inteligencia Artificial — Proyecto final, Opción A (sistema RAG)**

Los estudiantes suelen tener dudas sobre cancelaciones, promedios, materias perdidas, supletorios o devoluciones de dinero. La respuesta está en reglamentos extensos en PDF. Este asistente responde con base **solo** en esos reglamentos, muestra el artículo y la página en que se apoya y dice cuándo la información no está en los documentos.

**Stack:** PyMuPDF · LangChain · NVIDIA NIM · Qdrant (en memoria) · Gradio (opcional)

| Parte | Tarea |
|---|---|
| 0 | Instalar dependencias y configurar la clave de NVIDIA |
| 1 | Obtener los reglamentos en PDF |
| 2 | Extraer el texto y dividirlo por artículos |
| 3 | Generar embeddings e indexar en Qdrant |
| 4 | Pipeline RAG: recuperar, responder y citar fuentes |
| 5 | Cinco casos de prueba |
| 6 | Exploración del umbral de similitud |
| 7 | Interfaz Gradio (opcional) |

**Modelos (los mismos del curso):**
- Embeddings: `nvidia/nemotron-3-embed-1b`
- Generación: `nvidia/nemotron-3-super-120b-a12b`

In [1]:
# Dependencias (Google Colab o entorno local)
%pip install -qU pymupdf langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv pandas requests

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


---
## Parte 0 — Configurar la clave de NVIDIA NIM

La clave **nunca** se escribe en el notebook. Hay tres formas de darla, en este orden:

1. **Colab:** abre el panel 🔑 *Secrets*, crea un secreto llamado `NVIDIA_API_KEY` y habilita el acceso para este notebook.
2. **Local:** crea un archivo `.env` (a partir de `.env.example`) con `NVIDIA_API_KEY=...`. El archivo `.env` está en `.gitignore`.
3. Si no encuentra ninguna, la celda la pide de forma oculta con `getpass`.

La clave se obtiene gratis en https://build.nvidia.com/.

In [20]:
import os
import re
import getpass
from pathlib import Path
from typing import List

import requests
from dotenv import load_dotenv

load_dotenv()

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
    EN_COLAB = True
except (ImportError, KeyError):
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
    EN_COLAB = False
except Exception:
    # El secreto existe en Colab pero no se le dio acceso a este notebook.
    NVIDIA_API_KEY = None
    EN_COLAB = True

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY. Créala en https://build.nvidia.com/")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
GENERATION_MODEL = "nvidia/nemotron-3.5-super-vl-120b-a12b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"

print("✅ Configuración NVIDIA NIM lista")

NVIDIA_API_KEY: ··········
✅ Configuración NVIDIA NIM lista


---
## Parte 1 — Los documentos

Usamos dos reglamentos oficiales y públicos de EAFIT:

| Documento | Archivo | Páginas | Fuente oficial |
|---|---|---|---|
| Reglamento Académico de los Programas de Pregrado (versión actualizada a dic. de 2025) | `docs/reglamento-academico-pregrado.pdf` | 79 | [Página del reglamento](https://www.eafit.edu.co/institucional/reglamentos/reglamento-academico-de-los-programas-de-pregrado) |
| Reglamento Económico de la Relación Académica del Estudiante, versión VI | `docs/reglamento-economico.pdf` | 10 | [PDF en el sitio de EAFIT](https://universidadeafit.widen.net/s/wggqcsdnhf/reglamento_economico) |

Los PDF están incluidos en la carpeta `docs/` del repositorio. Si el notebook se abre en Colab sin la carpeta, la celda los descarga desde el repositorio de GitHub.

In [3]:
DOCS_DIR = Path("docs")
DOCS_DIR.mkdir(parents=True, exist_ok=True)

# Copia de los PDF en el repositorio del proyecto (para Colab).
REPO_RAW_URL = "https://raw.githubusercontent.com/saracastrillons/rag-reglamentos-eafit/main/docs"

ARCHIVOS = ["reglamento-academico-pregrado.pdf", "reglamento-economico.pdf"]


def obtener_pdf(nombre: str) -> None:
    destino = DOCS_DIR / nombre
    if destino.exists() and destino.stat().st_size > 10_000:
        print(f"↪ Ya existe: {nombre}")
        return
    respuesta = requests.get(f"{REPO_RAW_URL}/{nombre}", timeout=90)
    respuesta.raise_for_status()
    if not respuesta.content.startswith(b"%PDF"):
        raise ValueError(f"La descarga de {nombre} no produjo un PDF válido")
    destino.write_bytes(respuesta.content)
    print(f"⬇ Descargado: {nombre} ({destino.stat().st_size / 1e6:.1f} MB)")


for nombre in ARCHIVOS:
    try:
        obtener_pdf(nombre)
    except Exception as error:
        print(f"⚠ No se pudo obtener {nombre}: {error}")
        if EN_COLAB:
            from google.colab import files
            print(f"Sube manualmente {nombre} (está en la carpeta docs/ del repositorio):")
            for subido, contenido in files.upload().items():
                (DOCS_DIR / subido).write_bytes(contenido)

⬇ Descargado: reglamento-academico-pregrado.pdf (1.2 MB)
⬇ Descargado: reglamento-economico.pdf (0.1 MB)


---
## Parte 2 — Extraer el texto y dividirlo por artículos

Cambios respecto al workshop de papers, pensados para que cada respuesta pueda citar **artículo y página**:

- **PyMuPDF en lugar de MarkItDown.** MarkItDown convierte el PDF completo y se pierde el número de página. Con PyMuPDF leemos página por página y quitamos encabezados y pies de página repetidos (por ejemplo "Página 3 de 10").
- **Primero se corta por artículo y después por tamaño.** Un artículo trata un solo tema, que es lo que la clase recomienda para que el vector sea representativo. Solo se aceptan números consecutivos (1, 2, 3…) para no confundir un artículo nuevo con una referencia a otro artículo dentro del texto.
- **Fragmentos de hasta 1000 caracteres con solapamiento de 150.** Es algo más que los 400–600 sugeridos en clase porque los artículos tienen literales y parágrafos que pierden sentido si se separan. Los artículos largos se cortan preferiblemente en los "Parágrafo".
- **Encabezado en cada fragmento.** Cada fragmento empieza con `[Documento · Artículo N: título]`, así hasta el segundo pedazo de un artículo largo sabe de qué trata.

In [4]:
import pymupdf
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

DOCUMENTOS = {
    "reglamento-academico-pregrado.pdf": {
        "titulo": "Reglamento Académico de Pregrado",
        # Los artículos empiezan así: "Artículo 50. Cancelación de créditos académicos."
        "patron_articulo": r"Art[íi]culo\s+(\d+)\.",
        "ruido": [
            r"^\s*Reglamento Académico de los Programas de Pregrado\s*$",
            r"^\s*\d{1,3}\s*$",  # número de página
        ],
    },
    "reglamento-economico.pdf": {
        "titulo": "Reglamento Económico",
        # Los artículos empiezan así: "Art. 14. CANCELACIÓN DE LOS DERECHOS DE MATRÍCULA."
        "patron_articulo": r"Art\.\s*(\d+)\.",
        "ruido": [
            r"^\s*Página \d+ de \d+\s*$",
            r"^\s*Reglamento Económico Académico-?\s*$",
            r"^\s*Versión VI Aprobado noviembre de 2020\s*$",
            r"^\s*2015\s*$",
        ],
    },
}


def limpiar_pagina(texto: str, ruido: List[str]) -> str:
    """Quita encabezados, pies de página y espacios repetidos."""
    lineas = [
        linea for linea in texto.splitlines()
        if not any(re.match(patron, linea) for patron in ruido)
    ]
    texto = " ".join(linea.strip() for linea in lineas if linea.strip())
    texto = texto.replace(" ", " ")
    return re.sub(r"\s+", " ", texto).strip()


def leer_pdf(nombre: str, config: dict):
    """Devuelve el texto completo y la posición (carácter) donde empieza cada página."""
    partes, inicios_pagina, posicion = [], [], 0
    with pymupdf.open(DOCS_DIR / nombre) as pdf:
        for numero, pagina in enumerate(pdf, start=1):
            texto = limpiar_pagina(pagina.get_text(), config["ruido"])
            inicios_pagina.append((posicion, numero))
            partes.append(texto)
            posicion += len(texto) + 1
    return " ".join(partes), inicios_pagina


def pagina_en(posicion: int, inicios_pagina) -> int:
    pagina = 1
    for inicio, numero in inicios_pagina:
        if inicio > posicion:
            break
        pagina = numero
    return pagina


def titulo_articulo(texto: str) -> str:
    """Toma el nombre del artículo: lo que sigue al número hasta el primer punto."""
    nombre = re.split(r"[.:]", texto, maxsplit=1)[0].strip()
    palabras = nombre.split()
    # En el reglamento económico el nombre va en mayúsculas y a veces sin punto.
    if palabras and palabras[0].isupper():
        palabras = [p for p in palabras if p.isupper() or p in {"DE", "LA", "Y"}][:12]
        nombre = " ".join(palabras).capitalize()
    return nombre[:90]


def dividir_en_articulos(texto: str, patron: str):
    """Corta el texto en artículos aceptando solo números consecutivos."""
    cortes, esperado = [], 1
    for coincidencia in re.finditer(patron, texto):
        if int(coincidencia.group(1)) == esperado:
            cortes.append((coincidencia.start(), esperado, coincidencia.end()))
            esperado += 1

    articulos = []
    if cortes and cortes[0][0] > 0:
        articulos.append({"numero": None, "titulo": "Preámbulo y definiciones",
                          "inicio": 0, "texto": texto[:cortes[0][0]].strip()})
    for i, (inicio, numero, fin_encabezado) in enumerate(cortes):
        fin = cortes[i + 1][0] if i + 1 < len(cortes) else len(texto)
        articulos.append({
            "numero": numero,
            "titulo": titulo_articulo(texto[fin_encabezado:fin_encabezado + 200]),
            "inicio": inicio,
            "texto": texto[inicio:fin].strip(),
        })
    return articulos


splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    # Cortes "cero ancho": antes de cada Parágrafo o después de un punto, sin perder texto.
    separators=[r"(?= Parágrafo)", r"(?<=\. )", r"(?<=; )", r"(?<= )"],
    is_separator_regex=True,
    add_start_index=True,
)


def crear_fragmentos(nombre: str, config: dict) -> List[Document]:
    texto, inicios_pagina = leer_pdf(nombre, config)
    fragmentos = []
    for articulo in dividir_en_articulos(texto, config["patron_articulo"]):
        if articulo["numero"]:
            etiqueta = f"Artículo {articulo['numero']}: {articulo['titulo']}"
        else:
            etiqueta = articulo["titulo"]
        for indice, parte in enumerate(splitter.create_documents([articulo["texto"]])):
            inicio = articulo["inicio"] + parte.metadata["start_index"]
            pagina = pagina_en(inicio, inicios_pagina)
            pagina_fin = pagina_en(inicio + len(parte.page_content), inicios_pagina)
            fragmentos.append(Document(
                page_content=f"[{config['titulo']} · {etiqueta}]\n{parte.page_content.strip()}",
                metadata={
                    "documento": config["titulo"],
                    "archivo": nombre,
                    "articulo": articulo["numero"],
                    "titulo_articulo": articulo["titulo"],
                    "pagina": str(pagina) if pagina == pagina_fin else f"{pagina}-{pagina_fin}",
                    "fragmento": indice,
                },
            ))
    return fragmentos

In [5]:
todos_los_fragmentos: List[Document] = []
for nombre, config in DOCUMENTOS.items():
    fragmentos = crear_fragmentos(nombre, config)
    articulos = {f.metadata["articulo"] for f in fragmentos if f.metadata["articulo"]}
    print(f"{config['titulo']:<34} → {len(articulos):>3} artículos, {len(fragmentos):>3} fragmentos")
    todos_los_fragmentos += fragmentos

largos = [len(f.page_content) for f in todos_los_fragmentos]
print(f"\nTotal: {len(todos_los_fragmentos)} fragmentos · largo promedio {sum(largos) // len(largos)} caracteres")

ejemplo = next(f for f in todos_los_fragmentos if f.metadata["articulo"] == 50)
print("\nEjemplo de fragmento:")
print(ejemplo.metadata)
print(ejemplo.page_content[:500], "...")

Reglamento Académico de Pregrado   → 166 artículos, 337 fragmentos
Reglamento Económico               →  26 artículos,  39 fragmentos

Total: 376 fragmentos · largo promedio 725 caracteres

Ejemplo de fragmento:
{'documento': 'Reglamento Académico de Pregrado', 'archivo': 'reglamento-academico-pregrado.pdf', 'articulo': 50, 'titulo_articulo': 'Cancelación de créditos académicos', 'pagina': '23-24', 'fragmento': 0}
[Reglamento Académico de Pregrado · Artículo 50: Cancelación de créditos académicos]
Artículo 50. Cancelación de créditos académicos. Durante cada período académico, los estudiantes de pregrado podrán cancelar créditos académicos bajo las siguientes condiciones: 1. Límite de cancelación parcial: se podrá cancelar hasta el 50% de los créditos matriculados en el período académico. 2. Condición de evaluación: una asignatura podrá cancelarse si no ha sido evaluada en más del 75% de su programación,  ...


---
## Parte 3 — Embeddings NVIDIA e índice Qdrant

Igual que en el workshop: `NVIDIAEmbeddings` distingue entre embeddings de documentos y de consultas, la dimensión se obtiene del modelo y Qdrant se ejecuta en memoria. Al reiniciar el runtime hay que volver a indexar, lo que tarda alrededor de un minuto.

In [6]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

COLLECTION = "reglamentos_eafit"

embeddings = NVIDIAEmbeddings(model=EMBEDDING_MODEL, truncate="END")
EMBEDDING_DIM = len(embeddings.embed_query("prueba de dimensión"))

qdrant_client = QdrantClient(":memory:")
qdrant_client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
)

vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION,
    embedding=embeddings,
)
ids = vectorstore.add_documents(todos_los_fragmentos)

print(f"✅ {len(ids)} fragmentos indexados en Qdrant")
print(f"✅ Dimensión de embeddings: {EMBEDDING_DIM}")

/usr/local/lib/python3.13/dist-packages/langchain_nvidia_ai_endpoints/_common.py:269: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


✅ 376 fragmentos indexados en Qdrant
✅ Dimensión de embeddings: 2048


In [19]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA

PREGUNTA = "Responde en una sola frase en español: ¿qué es un sistema RAG?"

for modelo in ["nvidia/nemotron-3.5-lightning-30b-a3b", "nvidia/nemotron-3.5-super-vl-120b-a12b"]:
    llm = ChatNVIDIA(model=modelo, max_completion_tokens=1500)
    for etiqueta, extra in [("con razonamiento", {}),
                            ("sin razonamiento", {"chat_template_kwargs": {"enable_thinking": False}})]:
        try:
            r = llm.invoke(PREGUNTA, **extra)
            print("✅", modelo, "|", etiqueta, "→", repr(r.content.strip()[:150]))
        except Exception as e:
            print("❌", modelo, "|", etiqueta, "→", str(e).splitlines()[0][:80])

✅ nvidia/nemotron-3.5-lightning-30b-a3b | con razonamiento → 'Un sistema RAG (Generación Aumentada por Recuperación) es una arquitectura de IA que combina un modelo de lenguaje con una base de datos externa para '
✅ nvidia/nemotron-3.5-lightning-30b-a3b | sin razonamiento → 'Un sistema RAG es una arquitectura de inteligencia artificial que combina la recuperación de información con la generación de texto para proporcionar '
✅ nvidia/nemotron-3.5-super-vl-120b-a12b | con razonamiento → 'Un sistema RAG (Retrieval-Augmented Generation) es una arquitectura de IA que combina la recuperación de información externa con la generación de leng'
✅ nvidia/nemotron-3.5-super-vl-120b-a12b | sin razonamiento → 'Es una arquitectura de inteligencia artificial que combina la recuperación de información externa con la generación de texto para ofrecer respuestas m'


---
## Parte 4 — Pipeline RAG con fuentes y reconocimiento de contexto insuficiente

El sistema reconoce que no puede responder en **dos niveles**:

1. **Antes del LLM:** se recuperan los `K` fragmentos más parecidos con su similitud del coseno. Si ninguno supera `UMBRAL`, no se llama al modelo y se responde que no hay información.
2. **En el LLM:** el prompt le exige usar solo los fragmentos y, si no alcanzan, responder con una frase fija (`NO_INFO`). Con esa frase las pruebas pueden comprobar automáticamente si el sistema se abstuvo.

Cada respuesta va acompañada de sus fuentes: documento, artículo, página, similitud y un extracto del fragmento.

In [21]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

K = 5            # fragmentos que se recuperan
UMBRAL = 0.25    # similitud mínima para usar un fragmento (ver Parte 6)
NO_INFO = "No encuentro información suficiente en los reglamentos para responder esa pregunta."

llm = ChatNVIDIA(
    model=GENERATION_MODEL,
    temperature=0.1,
    top_p=0.95,
    max_completion_tokens=1500,
).bind(chat_template_kwargs={"enable_thinking": False})

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Eres un asistente que orienta a estudiantes de pregrado de la Universidad EAFIT "
        "sobre sus reglamentos. Sigue estas reglas:\n"
        "1. Responde únicamente con la información de los FRAGMENTOS. No uses conocimiento "
        "propio ni normas de otras universidades.\n"
        "2. Después de cada afirmación cita la fuente entre corchetes, por ejemplo "
        "[Reglamento Académico de Pregrado, Art. 50].\n"
        "3. Si los fragmentos no contienen la información necesaria, responde exactamente: "
        "\"{no_info}\" y en una frase di qué tema no aparece.\n"
        "4. Si solo puedes responder una parte de la pregunta, responde esa parte y aclara "
        "qué parte no está en los reglamentos.\n"
        "5. Responde en español, de forma clara y breve (máximo 6 oraciones o una lista corta).",
    ),
    ("human", "FRAGMENTOS:\n{contexto}\n\nPREGUNTA:\n{pregunta}"),
]).partial(no_info=NO_INFO)

cadena_respuesta = prompt | llm | StrOutputParser()


def recuperar(pregunta: str):
    """Devuelve [(fragmento, similitud)] ordenados de mayor a menor similitud."""
    return vectorstore.similarity_search_with_score(pregunta, k=K)


def formatear_contexto(resultados) -> str:
    return "\n\n---\n\n".join(
        f"[Fuente {i}] {doc.metadata['documento']}, página {doc.metadata['pagina']}\n{doc.page_content}"
        for i, (doc, _) in enumerate(resultados, start=1)
    )


def cita(doc: Document) -> str:
    m = doc.metadata
    articulo = f"Art. {m['articulo']} ({m['titulo_articulo']})" if m["articulo"] else m["titulo_articulo"]
    return f"{m['documento']} · {articulo} · p. {m['pagina']}"


def preguntar(pregunta: str) -> dict:
    recuperados = recuperar(pregunta)
    relevantes = [(doc, score) for doc, score in recuperados if score >= UMBRAL]

    if not relevantes:
        respuesta = NO_INFO + " (Ningún fragmento superó el umbral de similitud.)"
    else:
        respuesta = cadena_respuesta.invoke({
            "contexto": formatear_contexto(relevantes),
            "pregunta": pregunta,
        })
        # Algunos modelos razonadores devuelven su razonamiento entre <think>...</think>.
        respuesta = re.sub(r"<think>.*?</think>", "", respuesta, flags=re.DOTALL).strip()

    return {
        "pregunta": pregunta,
        "respuesta": respuesta,
        "fuentes": relevantes,
        "descartados": [(doc, score) for doc, score in recuperados if score < UMBRAL],
        "se_abstuvo": NO_INFO.lower().rstrip(".") in respuesta.lower(),
    }


def mostrar(resultado: dict, largo_extracto: int = 220) -> None:
    print(f"❓ {resultado['pregunta']}\n")
    print(f"💬 {resultado['respuesta']}\n")
    print("📚 Fuentes recuperadas:")
    if not resultado["fuentes"]:
        print("   (ninguna superó el umbral)")
    for doc, score in resultado["fuentes"]:
        extracto = doc.page_content.split("\n", 1)[-1][:largo_extracto].replace("\n", " ")
        print(f"   • {cita(doc)} · similitud {score:.2f}")
        print(f"     «{extracto}…»")
    if resultado["descartados"]:
        print(f"   ({len(resultado['descartados'])} fragmento(s) descartado(s) por debajo del umbral {UMBRAL})")

In [22]:
mostrar(preguntar("¿Cuál es la nota mínima para aprobar una materia?"))

❓ ¿Cuál es la nota mínima para aprobar una materia?

💬 La calificación definitiva cuantitativa mínima para aprobar una asignatura de pregrado es tres punto cero (3.0) [Reglamento Académico de Pregrado, Artículo 76]. También se puede aprobar con una calificación cualitativa de A: Aprobada [Reglamento Académico de Pregrado, Artículo 76].

📚 Fuentes recuperadas:
   • Reglamento Académico de Pregrado · Art. 76 (Escala de calificaciones) · p. 37 · similitud 0.41
     «Parágrafo 1. Para todos los efectos académicos una asignatura de pregrado se considera aprobada si al final de ésta se obtuvo una calificación definitiva cuantitativa de tres punto cero (3.0) o superior, o una calificaci…»
   • Reglamento Económico · Art. 18 (De los exámenes de validación) · p. 9 · similitud 0.28
     «Art. 18. DE LOS EXÁMENES DE VALIDACIÓN: El estudiante al cual le sea aprobada la presentación de un examen de suficiencia, deberá registrar y pagar el valor de la materia correspondiente.…»
   • Reglamento Acadé

---
## Parte 5 — Casos de prueba

Cinco casos: tres habituales, uno que exige combinar los dos reglamentos y un caso límite con información ausente. Para cada caso se registra la respuesta esperada, tomada de leer directamente el reglamento, y los artículos donde está.

Además de mostrar la respuesta, la celda calcula dos comprobaciones objetivas:
- **¿Recuperó el artículo esperado?** Indica si los fragmentos usados incluyen el artículo donde está la respuesta.
- **¿Se abstuvo?** Indica si el sistema dijo que no tenía información. Solo debería ocurrir en el caso límite.

La **valoración** final es manual: comparar la respuesta con el texto del reglamento y revisar que cada afirmación esté respaldada por los fragmentos mostrados.

In [9]:
ACAD = "Reglamento Académico de Pregrado"
ECON = "Reglamento Económico"

CASOS = [
    {
        "id": 1,
        "tipo": "Habitual",
        "pregunta": "¿Qué porcentaje de los créditos matriculados puedo cancelar en un semestre y qué condiciones hay?",
        "esperado": "Hasta el 50 % de los créditos matriculados. La asignatura no debe haber sido evaluada en más del 75 % "
                    "y la cancelación se hace dentro de la fecha límite del calendario académico. No se pueden cancelar "
                    "créditos de práctica ni asignaturas con matrícula condicional, salvo fuerza mayor ante el Comité de Casos.",
        "articulos_esperados": [(ACAD, 50)],
    },
    {
        "id": 2,
        "tipo": "Habitual",
        "pregunta": "¿Qué pasa si pierdo la misma materia por tercera vez?",
        "esperado": "Debe matricularla por cuarta vez en el semestre siguiente, con matrícula condicional y un máximo de "
                    "12 créditos. Durante la 3.ª, 4.ª y 5.ª matrícula no puede cancelarla. Si la pierde en la cuarta "
                    "matrícula, queda retirado por rendimiento académico.",
        "articulos_esperados": [(ACAD, 51)],
    },
    {
        "id": 3,
        "tipo": "Habitual",
        "pregunta": "¿Con qué promedio me retiran de la universidad por bajo rendimiento académico?",
        "esperado": "Depende del número de matrículas: primera, promedio semestral inferior a 2.5; segunda o tercera, "
                    "promedio semestral y acumulado inferior a 2.8; cuarta en adelante, inferior a 3.0. También se retira "
                    "a quien pierde una materia matriculada por cuarta vez.",
        "articulos_esperados": [(ACAD, 52)],
    },
    {
        "id": 4,
        "tipo": "Cruza los dos documentos",
        "pregunta": "Si cancelo una materia durante la segunda semana de clases, ¿me devuelven el dinero?",
        "esperado": "El Reglamento Académico permite cancelar créditos (Art. 50) y remite al Reglamento Económico para la "
                    "devolución. Este dice: antes de iniciar la 2.ª semana, 90 %; antes de terminar la 2.ª semana, 75 %; "
                    "desde la 3.ª semana, nada. Los estudiantes de primer semestre no pueden retirar materias.",
        "articulos_esperados": [(ACAD, 50), (ECON, 17)],
    },
    {
        "id": 5,
        "tipo": "Límite (información ausente)",
        "pregunta": "¿Cuánto cuesta el parqueadero de la universidad para estudiantes?",
        "esperado": "Los reglamentos no hablan del parqueadero. El sistema debe decir que no tiene esa información y no "
                    "inventar una tarifa.",
        "articulos_esperados": [],
    },
]

In [23]:
import pandas as pd

filas = []
for caso in CASOS:
    print("=" * 100)
    print(f"CASO {caso['id']} · {caso['tipo']}\n")
    resultado = preguntar(caso["pregunta"])
    mostrar(resultado)
    print(f"\n✅ Esperado: {caso['esperado']}")

    recuperados = {(d.metadata["documento"], d.metadata["articulo"]) for d, _ in resultado["fuentes"]}
    esperados = set(caso["articulos_esperados"])
    if esperados:
        encontrados = esperados & recuperados
        recupero = "Sí" if encontrados == esperados else ("Parcial" if encontrados else "No")
    else:
        recupero = "N/A"

    filas.append({
        "Caso": caso["id"],
        "Tipo": caso["tipo"],
        "Pregunta": caso["pregunta"],
        "Resultado esperado": caso["esperado"],
        "Respuesta obtenida": resultado["respuesta"],
        "Fuentes": "; ".join(cita(d) for d, _ in resultado["fuentes"]),
        "¿Recuperó el artículo esperado?": recupero,
        "¿Se abstuvo?": "Sí" if resultado["se_abstuvo"] else "No",
    })

resultados = pd.DataFrame(filas)
resultados[["Caso", "Tipo", "¿Recuperó el artículo esperado?", "¿Se abstuvo?"]]

CASO 1 · Habitual

❓ ¿Qué porcentaje de los créditos matriculados puedo cancelar en un semestre y qué condiciones hay?

💬 Puedes cancelar hasta el 50% de los créditos matriculados en el período académico [Reglamento Académico de Pregrado, Art. 50]. Las condiciones son: la asignatura no debe haber sido evaluada en más del 75% de su programación [Reglamento Académico de Pregrado, Art. 50]; la cancelación debe hacerse dentro de la fecha límite del calendario académico [Reglamento Académico de Pregrado, Art. 50]; los créditos de práctica no se pueden cancelar salvo por fuerza mayor o caso fortuito comprobado ante el Comité de Casos [Reglamento Académico de Pregrado, Art. 50]; y los estudiantes con matrícula condicional no pueden cancelar asignaturas, salvo por fuerza mayor o caso fortuito comprobado ante el Comité de Casos [Reglamento Académico de Pregrado, Art. 50].

📚 Fuentes recuperadas:
   • Reglamento Académico de Pregrado · Art. 50 (Cancelación de créditos académicos) · p. 23-24 · si

,Caso,Tipo,¿Recuperó el artículo esperado?,¿Se abstuvo?
0,1,Habitual,Sí,No
1,2,Habitual,Sí,No
2,3,Habitual,Sí,No
3,4,Cruza los dos documentos,Parcial,No
4,5,Límite (información ausente),N/A,Sí


In [24]:
# Guarda los resultados para copiarlos al README.
Path("resultados").mkdir(exist_ok=True)
resultados.to_csv("resultados/resultados_pruebas.csv", index=False, encoding="utf-8-sig")


def a_markdown(df: pd.DataFrame) -> str:
    limpiar = lambda valor: str(valor).replace("|", "/").replace("\n", " ")
    lineas = ["| " + " | ".join(df.columns) + " |", "|" + "---|" * len(df.columns)]
    lineas += ["| " + " | ".join(limpiar(v) for v in fila) + " |" for fila in df.itertuples(index=False)]
    return "\n".join(lineas)


Path("resultados/resultados_pruebas.md").write_text(a_markdown(resultados), encoding="utf-8")
print("✅ Guardado en resultados/resultados_pruebas.csv y resultados/resultados_pruebas.md")

✅ Guardado en resultados/resultados_pruebas.csv y resultados/resultados_pruebas.md


### Valoración (completar después de ejecutar)

Para cada caso, abran el artículo citado en el PDF y respondan:

- ¿La respuesta es **relevante**, es decir, atiende lo que se preguntó?
- ¿Cada afirmación está **respaldada** por los fragmentos mostrados (fidelidad)?
- ¿Las citas de artículo y página son correctas?
- ¿Falta algo importante que sí está en el reglamento?

Las valoraciones y el análisis de la limitación van en el `README.md`.

---
## Parte 6 — Exploración del umbral de similitud

El valor de `UMBRAL` no es universal: depende del modelo de embeddings. Esta celda compara las similitudes de preguntas que **sí** están en los reglamentos con las de preguntas que **no** están, para justificar el valor elegido.

La última pregunta es un caso difícil: el reglamento económico habla mucho de la matrícula, pero no dice cuánto cuesta. Los fragmentos recuperados tendrán similitud alta aunque no contengan la respuesta, así que el umbral no basta y la abstención depende del LLM.

In [25]:
PREGUNTAS_EXPLORACION = [
    ("Dentro del dominio", "¿Cuántas veces puedo matricular una misma asignatura?"),
    ("Dentro del dominio", "¿Los exámenes supletorios tienen algún costo?"),
    ("Fuera del dominio", "¿Cuál es la receta de la bandeja paisa?"),
    ("Fuera del dominio", "¿Cuánto cuesta el parqueadero de la universidad para estudiantes?"),
    ("Difícil", "¿Cuánto vale la matrícula de Ingeniería de Sistemas?"),
]

for tipo, pregunta in PREGUNTAS_EXPLORACION:
    puntajes = [score for _, score in recuperar(pregunta)]
    print(f"{tipo:<19} máx {max(puntajes):.2f} · mín {min(puntajes):.2f} · {pregunta}")

Dentro del dominio  máx 0.65 · mín 0.40 · ¿Cuántas veces puedo matricular una misma asignatura?
Dentro del dominio  máx 0.50 · mín 0.35 · ¿Los exámenes supletorios tienen algún costo?
Fuera del dominio   máx 0.07 · mín 0.05 · ¿Cuál es la receta de la bandeja paisa?
Fuera del dominio   máx 0.26 · mín 0.24 · ¿Cuánto cuesta el parqueadero de la universidad para estudiantes?
Difícil             máx 0.28 · mín 0.25 · ¿Cuánto vale la matrícula de Ingeniería de Sistemas?


In [26]:
# El caso difícil completo: ¿el LLM reconoce que no hay valores de matrícula?
mostrar(preguntar("¿Cuánto vale la matrícula de Ingeniería de Sistemas?"))

❓ ¿Cuánto vale la matrícula de Ingeniería de Sistemas?

💬 No encuentro información suficiente en los reglamentos para responder esa pregunta. No aparece el valor de la matrícula de Ingeniería de Sistemas.

📚 Fuentes recuperadas:
   • Reglamento Económico · Art. 7 (De los componentes de la matrícula) · p. 4 · similitud 0.28
     «Art. 7. DE LOS COMPONENTES DE LA MATRÍCULA. En el valor de la matrícula, están incluidos los siguientes rubros: a. Matrícula por créditos o por UMES (Unidad de Matrícula en EAFIT): Corresponde al valor específico de las …»
   • Reglamento Académico de Pregrado · Art. 38 (Matrícula) · p. 20 · similitud 0.26
     «Parágrafo 5. Para que un estudiante pueda realizar el proceso de matrícula debe encontrarse a paz y salvo con la Universidad EAFIT por todo concepto. Parágrafo 6. La matrícula para un período académico se perfecciona con…»
   • Reglamento Económico · Art. 13 (Base para la liquidación matrícula) · p. 6 · similitud 0.25
     «Art. 13. BASE PARA LA LIQUIDA

---
## Parte 7 — Interfaz Gradio (opcional)

Adaptada del notebook `02b` del curso. En Colab, `share=True` crea un enlace público temporal. Para detener la aplicación, interrumpe la celda.

In [13]:
%pip install -qU gradio

In [14]:
import gradio as gr


def chat(pregunta: str, historial: list | None):
    historial = historial or []
    pregunta = (pregunta or "").strip()
    if not pregunta:
        return "", historial, "Escribe una pregunta."

    try:
        resultado = preguntar(pregunta)
        respuesta = resultado["respuesta"]
        if resultado["fuentes"]:
            fuentes_md = "**Fuentes recuperadas:**\n" + "\n".join(
                f"- {cita(doc)} · similitud {score:.2f}" for doc, score in resultado["fuentes"]
            )
        else:
            fuentes_md = "Ningún fragmento de los reglamentos superó el umbral de similitud."
    except Exception as error:
        respuesta = "Ocurrió un error al consultar el pipeline."
        fuentes_md = f"**Error:** `{type(error).__name__}: {error}`"

    historial = historial + [
        {"role": "user", "content": pregunta},
        {"role": "assistant", "content": respuesta},
    ]
    return "", historial, fuentes_md


with gr.Blocks(title="Asistente de reglamentos EAFIT") as demo:
    gr.Markdown(
        "# Asistente de reglamentos de pregrado — EAFIT\n"
        "Responde con base en el Reglamento Académico de Pregrado y el Reglamento Económico. "
        "Orienta, pero no reemplaza a Admisiones y Registro."
    )
    chatbot = gr.Chatbot(height=420, label="Chat")
    with gr.Row():
        caja = gr.Textbox(placeholder="Escribe tu pregunta sobre los reglamentos...",
                          label="Pregunta", lines=2, scale=4)
        enviar = gr.Button("Enviar", variant="primary", scale=1)
    fuentes = gr.Markdown()
    gr.Examples(
        examples=[[caso["pregunta"]] for caso in CASOS],
        inputs=caja,
        label="Preguntas de ejemplo",
    )
    enviar.click(chat, inputs=[caja, chatbot], outputs=[caja, chatbot, fuentes])
    caja.submit(chat, inputs=[caja, chatbot], outputs=[caja, chatbot, fuentes])

demo.queue().launch(share=EN_COLAB, debug=True, show_error=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://17e433887eabcf2769.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://17e433887eabcf2769.gradio.live
